## Text input

https://platform.openai.com/docs/models

In [4]:
from dotenv import load_dotenv

load_dotenv()

True

In [5]:
from langchain.agents import create_agent

agent = create_agent(
    model='google_genai:gemini-3.1-flash-lite',
    system_prompt="You are a science fiction writer, create a capital city at the users request.",
)


In [6]:
from langchain.messages import HumanMessage

question = HumanMessage(content=[
    {"type": "text", "text": "What is the capital of The Moon?"}
])

response = agent.invoke(
    {"messages": [question]}
)

print(response['messages'][-1].content)

[{'type': 'text', 'text': 'The capital of the Moon is **Artemis Prime**, a sprawling, multi-layered subterranean megalopolis located beneath the basalt plains of the **Mare Tranquillitatis**.\n\nBecause the lunar surface is hostile—punctuated by extreme radiation, micrometeoroid impacts, and two-week-long nights—Artemis Prime is not a city of skyscrapers, but a city of *geodesic ventricles* and vertical shafts carved deep into the lunar crust.\n\nHere is a glimpse into the capital:\n\n### 1. The Architecture of "The Hive"\nArtemis Prime is built inside a massive, naturally occurring lava tube. The city architecture is bioluminescent and modular. The walls of the tube are reinforced with "Lunar-Crete"—a mixture of regolith and polymer resin—and lined with millions of hexagonal apartments that glow a soft amber. From a distance, the city looks like a glowing coral reef buried in the dark, silent heart of the Moon.\n\n### 2. The Zenith Plaza\nThe heart of the city is the **Zenith Plaza**,

## Image input

In [7]:
from ipywidgets import FileUpload
from IPython.display import display

uploader = FileUpload(accept='.png', multiple=False)
display(uploader)

FileUpload(value=(), accept='.png', description='Upload')

In [8]:
print(uploader.value)

({'name': 'home-burger.png', 'type': 'image/png', 'size': 297001, 'content': <memory at 0x0000018CD22F9A80>, 'last_modified': datetime.datetime(2024, 6, 17, 12, 46, 7, tzinfo=datetime.timezone.utc)},)


In [9]:
import base64

# Get the first (and only) uploaded file dict
uploaded_file = uploader.value[0]

# This is a memoryview
content_mv = uploaded_file["content"]

# Convert memoryview -> bytes
img_bytes = bytes(content_mv)  # or content_mv.tobytes()

# Now base64 encode
img_b64 = base64.b64encode(img_bytes).decode("utf-8")

In [11]:
multimodal_question = HumanMessage(content=[
    {"type": "text", "text": "Tell me about this image"},
    {"type": "image", "base64": img_b64, "mime_type": "image/png"}
])

response = agent.invoke(
    {"messages": [multimodal_question]}
)

print(response['messages'][-1].content)

[{'type': 'text', 'text': 'As a science fiction writer, I don’t just see a double cheeseburger. I see the blueprint for **Aethelgard Prime**, the capital city of the culinary quadrant of the Andromeda frontier.\n\n***\n\n### Aethelgard Prime: The City of Layers\n\nAethelgard Prime is not built on a planet, but on a massive, slow-rotating orbital plate known as "The Bun." The city is famous for its stratified verticality, where civilization is arranged in succulent, habitable tiers.\n\n**1. The Sesame Crust (The Upper Atmospheric Domes)**\nThe outermost layer of the city, known as The Crust, is a glittering expanse of white, domed bio-habitats that look like floating seeds from space. This is the exclusive residential district for the elite, where solar-tempered air creates a perpetual, golden-hour glow.\n\n**2. The Verdant Canopy (The Agricultural District)**\nBeneath the Crust lies a sprawling, emerald-green forest of genetically enhanced hydroponic lettuce. This isn\'t just decoratio

## Audio input

In [12]:
import sounddevice as sd
from scipy.io.wavfile import write
import base64
import io
import time
from tqdm import tqdm

# Recording settings
duration = 5  # seconds
sample_rate = 44100

print("Recording...")
audio = sd.rec(int(duration * sample_rate), samplerate=sample_rate, channels=1)
# Progress bar for the duration
for _ in tqdm(range(duration * 10)):   # update 10× per second
    time.sleep(0.1)
sd.wait()
print("Done.")

# Write WAV to an in-memory buffer
buf = io.BytesIO()
write(buf, sample_rate, audio)
wav_bytes = buf.getvalue()

aud_b64 = base64.b64encode(wav_bytes).decode("utf-8")

Recording...


100%|██████████| 50/50 [00:05<00:00,  9.84it/s]

Done.


In [14]:
agent = create_agent(
    model='google_genai:gemini-3.1-flash-lite',
)

multimodal_question = HumanMessage(content=[
    {"type": "text", "text": "Tell me about this audio file"},
    {"type": "audio", "base64": aud_b64, "mime_type": "audio/wav"}
])

response = agent.invoke(
    {"messages": [multimodal_question]}
)

print(response['messages'][-1].content)

[{'type': 'text', 'text': 'The audio contains the sound of a chime.', 'extras': {'signature': 'EnMKcQFpFH0T8bx0U/aFzEHngCMSq/YVCHbEtY8C0e9CjL19QaIy8kG5QBG2/6QzMcxVZXAZNdDh2HMCL7IoGN8bHrOf0aCck1Uk5vXvSLFv4AelJfUBqGhW5hz8TH4FOP1fAACmIlWN2L1d85/vL6HDWJwZ'}}]
